In [4]:
import numpy as np
import torch.nn as nn
import torch
import gymnasium as gym

from torch.distributions import Categorical
import torch.nn.functional as F

In [ ]:
# VARIABLES
OBSERVATION_DIM = 4
ACTION_DIM = 2

NUM_ENVS = 32
TRAJECTORY_WINDOW = 128

CLIP = 0.2
VF_COEF = 0.5
ENT_COEF = 0.01
EPOCHS = 10
MB_SIZE = 256

GAMMA = 0.99
LAMBDA = 0.95

NUM_EPISODES = 25

In [2]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__()
        # "new actor"
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, obs):
        action_probs = self.target_actor(obs)
        value = self.critic(obs)
        return action_probs, value

NameError: name 'nn' is not defined

In [ ]:
from core.environment import Environment
from core.configs import EnvironmentConfig
torch.random_seed(42)

cfg = EnvironmentConfig(num_agents=2, SEE_CARD_COUNTS=True)
env = Environment(cfg)

In [ ]:
ppomodel = ActorCritic()
criterion = nn.MSELoss()

In [ ]:
# Collect trajectories
observations = []
actions = []
rewards = []
values = []

obs, _ = env.reset()
for episode in range(NUM_EPISODES):
    with torch.no_grad():
        logits = ppomodel.actor(obs)
        action = F.softmax(logits)
        
        next_obs, rewards, terminated, truncated, info = env.step(obs)
